In [1]:
import numpy as np
import gapMetrics

In [2]:
#Simple SISO model
#continuous time state space model
# m:normalised mRNA
# M:normalised protein
# input(u):normalised input function

def input_func(u, form = "activation" ,params = None):
    # # Hill function
    # if form == "activation":
    #     u = alpha * u**n / (Kn**n + u**n)
    # elif form == "repression":
    #     u = alpha * Ku**n / (Ku**n + u**n)
    return u

def default_params(): 
    params = {} #parameters dictionary
    params['km'] = 0.7
    params['kM'] = 0.7
    params['deltam'] = 0.07
    params['deltaM'] = 0.03
    # #Hill function parameters
    # params['alpha'] = 1.0
    # params['n'] = 2.0
    # params['Kn'] = 0.5
    # params['Ku'] = 0.5
    return params

def model_dynamics(x, u, params):
    m = x[0]
    M = x[1]
    km = params['km']
    kM = params['kM']
    deltam = params['deltam']
    deltaM = params['deltaM']
    # u = input_func(u, form="activation", params=params)
    
    dm_dt = km + u - deltam * m
    dM_dt = kM * m - deltaM * M
    
    dxdt = np.array([dm_dt, dM_dt])
    return dxdt

def discretize_AB(delta_m, delta_M, T):
    """
    also normalised; x1 = m/km, x1 = M/(kM*km), u = 1+u/km
    then the model becomes:
    dxdt = [-deltam 0; 1 -deltaM]x + [1;0]u
    then discretise using ZOH with sample time T
    """
    em = np.exp(-delta_m * T)
    eM = np.exp(-delta_M * T)

    Ad = np.array([
        [em, 0.0],
        [(em - eM) / (delta_M - delta_m), eM]
    ])

    Bd = np.array([
        [(1 - em) / delta_m],
        [((1 - em) / delta_m - (1 - eM) / delta_M) / (delta_M - delta_m)]
    ])

    return Ad, Bd


In [ ]:
#call functions
params = default_params()
Ad1, Bd1 = discretize_AB(0.7, 0.03, 1.0)
# print("Discretized A matrix:\n", Ad)
# print("Discretized B matrix:\n", Bd)
Ad2, Bd2 = discretize_AB(0.6, 0.04, 1.0)
C = np.array([[0, 1]])
D = np.array([[0]])
num1, den1 = gapMetrics.ss_to_tf_discrete(Ad1, Bd1, C, D)
num2, den2 = gapMetrics.ss_to_tf_discrete(Ad2, Bd2, C, D)
v_gap = gapMetrics.vgap_metric(num1, den1, num2, den2)
print("v-gap between the two systems:", v_gap)
#For proportional controller with gain K
K = 1.0
num3 = np.empty((1, 1), dtype=object)
den3 = np.empty((1, 1), dtype=object)
num3[0, 0] = np.array([K])
den3[0, 0] = np.array([1.0])

bpc, _, _ = gapMetrics.vgap_bpc(num1, den1, num3, den3)
print("bpc:", bpc)

v-gap between the two systems: (0.04390122146430629, np.float64(0.04390122146430629))
bpc: 0.692104946462384
